In [1]:
!pip install mne pandas numpy scipy torch scikit-learn tqdm


In [2]:
# ============================================================
# FULL END-TO-END (fresh run; no caches assumed)
# Proposed model: CNN + BiLSTM on paired EEG+ECG
#
# Adds what you were missing:
#   (A) Saves SEQUENCE-LEVEL metrics + curves into OUT_DIR:
#       - val_seq_metrics.npz  (val_seq_auroc, val_seq_aupr)
#       - seq_roc_curve.npz    (fpr,tpr,thr)
#       - seq_pr_curve.npz     (precision,recall,thr)
#       - seq_ROC.png, seq_PR.png
#
#   (B) Saves EVENT-LEVEL curve CSVs + “paper-looking” plots:
#       - event_curve_episode_fah.csv
#       - event_curve_window_fah.csv
#       - episode_envelope_monotone.png
#
#   (C) Saves POINTWISE metrics + curves (from NPZ timepoints):
#       - pointwise_metrics_at_best_episode_thr.npz
#       - pointwise_ROC.png, pointwise_PR.png, prob_histogram.png
#
# Evaluation protocol (as in your last run):
# - 2s windows
# - PRE_TOL_SEC=30s for event detection
# - Detection WITHOUT refractory (raw alarm)
# - False-alarm EPISODE counting WITH refractory + episode gap
#
# NOTE: This will take time (cache build + training + inference).
# ============================================================

import os, re, gc, time, random
from glob import glob
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import mne
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)

import matplotlib.pyplot as plt

# =========================
# PATHS
# =========================
DATASET_PATH = "/kaggle/input/datasets/lohithmarneni/seizelt2full"
OUT_DIR = "/kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# CONFIG
# =========================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

mne.set_log_level("ERROR")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

SFREQ = 256
WIN_SEC = 2.0
WIN_SAMP = int(WIN_SEC * SFREQ)

# caches
TRAIN_MAX_WINDOWS = 300_000
VAL_MAX_WINDOWS   = 120_000
TARGET_POS_FRAC_TRAIN = 0.20
TARGET_POS_FRAC_VAL   = 0.20

BG_STEP_SEC = 2.0
SZ_STEP_SEC = 0.5

# baseline caps (isolate eval protocol)
POS_CAP_PER_RUN = 120
NEG_CAP_PER_RUN_WITH_SEIZURE = 600
NEG_CAP_PER_SEIZURE_FREE_RUN = 120

# filtering
APPLY_FILTER = True
EEG_BANDPASS = (1.0, 25.0)
ECG_BANDPASS = (0.5, 40.0)
APPLY_NOTCH_50 = False
MAX_PRELOAD_MB = 700.0

# channels
EEG_CH = 2
ECG_CH = 1
C_TOTAL = EEG_CH + ECG_CH

# training
BATCH_SIZE = 256
MAX_EPOCHS = 20
PATIENCE = 10
LR = 2e-4
WEIGHT_DECAY = 1e-4
MIN_DELTA = 5e-4
SCORE_LOSS_WEIGHT = 0.10

# sequence modeling
SEQ_LEN = 20
SEQ_POS_LAST_PROB = 0.60
SEQ_NEG_LAST_PROB = 0.10
DROP_AMBIGUOUS_SEQS = True

# run prob cache
PRED_STEP_SEC = 1.0

# alarm rules
ALARM_RULES = [
    (8,6), (10,8), (10,9),
    (20,14), (20,16), (20,18),
    (30,18), (30,24)
]
REFRACTORY_SEC = 60.0
EPISODE_GAP_SEC = 30.0

# eval protocol
PRE_TOL_SEC = 30.0
DET_USE_REFRACTORY = False   # detection uses raw alarm
FA_USE_REFRACTORY = True     # FA counting uses refractory alarm

THRESHOLDS = sorted(set(
    list(np.geomspace(1e-4, 5e-2, 20))
    + list(np.linspace(0.05, 0.95, 37))
))

FAH_LIMIT = 100.0
BG_ALARM_FRACTION_CAP = 0.30

# =========================
# OUTPUT FILES
# =========================
TR_X_MM = os.path.join(OUT_DIR, "X_train_mm.dat")
TR_Y_MM = os.path.join(OUT_DIR, "y_train_mm.dat")
VA_X_MM = os.path.join(OUT_DIR, "X_val_mm.dat")
VA_Y_MM = os.path.join(OUT_DIR, "y_val_mm.dat")

MODEL_PATH = os.path.join(OUT_DIR, "cnn_bilstm_best.pt")

# saved SEQ metrics/curves (what you need for paper row)
SEQ_METRICS_NPZ = os.path.join(OUT_DIR, "val_seq_metrics.npz")
SEQ_ROC_NPZ     = os.path.join(OUT_DIR, "seq_roc_curve.npz")
SEQ_PR_NPZ      = os.path.join(OUT_DIR, "seq_pr_curve.npz")
SEQ_ROC_PNG     = os.path.join(OUT_DIR, "seq_ROC.png")
SEQ_PR_PNG      = os.path.join(OUT_DIR, "seq_PR.png")

VAL_RUN_PROB_DIR = os.path.join(OUT_DIR, f"val_run_probs_step{str(PRED_STEP_SEC).replace('.','p')}")
os.makedirs(VAL_RUN_PROB_DIR, exist_ok=True)

CSV_WINDOW  = os.path.join(OUT_DIR, "event_curve_window_fah.csv")
CSV_EPISODE = os.path.join(OUT_DIR, "event_curve_episode_fah.csv")

EP_ENVELOPE_PNG = os.path.join(OUT_DIR, "episode_envelope_monotone.png")

# pointwise plots
PW_ROC_PNG  = os.path.join(OUT_DIR, "pointwise_ROC.png")
PW_PR_PNG   = os.path.join(OUT_DIR, "pointwise_PR.png")
PW_HIST_PNG = os.path.join(OUT_DIR, "pointwise_prob_histogram.png")
PW_METRICS_NPZ = os.path.join(OUT_DIR, "pointwise_metrics_at_best_episode_thr.npz")

# =========================
# PAIRING
# =========================
def get_subject_split(sub: str) -> Optional[str]:
    if not sub.startswith("sub-"): return None
    try:
        n = int(sub.split("-")[1])
    except Exception:
        return None
    if 1 <= n <= 96: return "train"
    if 97 <= n <= 125: return "val"
    return None

RUN_RE = re.compile(r"(sub-\d+)_.*_run-(\d+)_")

def parse_run_key(edf_path: str) -> Optional[Tuple[str,str]]:
    m = RUN_RE.search(os.path.basename(edf_path))
    if not m: return None
    return (m.group(1), m.group(2))

def collect_pairs(root: str):
    eeg_files = glob(os.path.join(root, "sub-*", "**", "*_eeg.edf"), recursive=True)
    ecg_files = glob(os.path.join(root, "sub-*", "**", "*_ecg.edf"), recursive=True)

    eeg_map={}
    for f in eeg_files:
        k=parse_run_key(f)
        if k: eeg_map[k]=f

    ecg_map={}
    for f in ecg_files:
        k=parse_run_key(f)
        if k: ecg_map[k]=f

    pairs=[]
    for k,eeg in eeg_map.items():
        if k in ecg_map:
            pairs.append((k[0], k[1], eeg, ecg_map[k]))
    pairs.sort()
    return pairs

pairs = collect_pairs(DATASET_PATH)
train_pairs = [p for p in pairs if get_subject_split(p[0]) == "train"]
val_pairs   = [p for p in pairs if get_subject_split(p[0]) == "val"]
print("Paired EEG+ECG runs:", len(pairs), "| train:", len(train_pairs), "| val:", len(val_pairs))
assert len(val_pairs) > 0

random.shuffle(train_pairs)
random.shuffle(val_pairs)

# =========================
# EVENTS
# =========================
def read_events_tsv(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
    except Exception:
        return pd.DataFrame()
    if "onset" not in df.columns or "duration" not in df.columns:
        return pd.DataFrame()
    if "eventType" not in df.columns:
        df["eventType"] = ""
    return df

def seizure_intervals(df: pd.DataFrame) -> List[Tuple[float,float]]:
    out=[]
    if df.empty: return out
    for _, r in df.iterrows():
        et = str(r.get("eventType","")).lower().strip()
        if not et.startswith("sz"):
            continue
        try:
            a=float(r["onset"]); d=float(r["duration"])
        except Exception:
            continue
        if d<=0: continue
        out.append((a, a+d))
    out.sort()
    return out

def overlaps_any(t0: float, t1: float, intervals: List[Tuple[float,float]]) -> bool:
    for a,b in intervals:
        if t0 < b and t1 > a:
            return True
    return False

# =========================
# PREPROCESS
# =========================
def zscore_clip(x: np.ndarray) -> np.ndarray:
    x=x.astype(np.float32, copy=False)
    x=(x-x.mean(axis=1, keepdims=True))/(x.std(axis=1, keepdims=True)+1e-6)
    return np.clip(x, -6.0, 6.0).astype(np.float32)

# =========================
# EDF IO + FILTER
# =========================
def preload_guard_read_raw(edf_path: str):
    hdr = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
    if int(hdr.info["sfreq"]) != SFREQ:
        hdr.close()
        raise ValueError("sfreq != 256")
    est_mb = (len(hdr.ch_names) * hdr.n_times * 8) / (1024**2)
    hdr.close()
    raw = mne.io.read_raw_edf(edf_path, preload=(est_mb <= MAX_PRELOAD_MB), verbose=False)
    return raw

def maybe_filter(raw: mne.io.BaseRaw, kind: str):
    if not APPLY_FILTER:
        return raw
    raw2 = raw.copy()
    if kind == "eeg":
        l,h = EEG_BANDPASS
    else:
        l,h = ECG_BANDPASS
    raw2.filter(l_freq=l, h_freq=h, method="iir", verbose=False)
    if APPLY_NOTCH_50:
        raw2.notch_filter(freqs=[50.0], method="iir", verbose=False)
    return raw2

def pick_eeg_picks(raw_eeg: mne.io.BaseRaw) -> List[int]:
    return list(range(min(EEG_CH, len(raw_eeg.ch_names))))

def pick_ecg_pick(raw_ecg: mne.io.BaseRaw) -> int:
    return 0

# =========================
# CACHE BUILDER
# =========================
def init_memmap_x(path: str, nmax: int) -> np.memmap:
    return np.memmap(path, mode="w+", dtype="float32", shape=(nmax, C_TOTAL, WIN_SAMP))

def init_memmap_y(path: str, nmax: int) -> np.memmap:
    return np.memmap(path, mode="w+", dtype="uint8", shape=(nmax,))

def sample_times(duration_sec: float, step_sec: float) -> np.ndarray:
    if duration_sec <= WIN_SEC:
        return np.array([], dtype=np.float32)
    return np.arange(0.0, duration_sec - WIN_SEC + 1e-9, step_sec, dtype=np.float32)

def sample_pos_times(intervals: List[Tuple[float,float]], duration_sec: float) -> np.ndarray:
    if len(intervals) == 0:
        return np.array([], dtype=np.float32)
    all_t=[]
    for a,b in intervals:
        start=max(0.0, a-1.0)
        end=min(duration_sec-WIN_SEC, b+1.0)
        if end<=start: continue
        all_t.append(np.arange(start, end+1e-9, SZ_STEP_SEC, dtype=np.float32))
    if not all_t: return np.array([], dtype=np.float32)
    t=np.unique(np.round(np.concatenate(all_t), 3))
    return t.astype(np.float32)

def sample_bg_times(intervals: List[Tuple[float,float]], duration_sec: float) -> np.ndarray:
    t = sample_times(duration_sec, BG_STEP_SEC)
    if len(t)==0: return t
    keep=[]
    for t0 in t:
        if not overlaps_any(float(t0), float(t0)+WIN_SEC, intervals):
            keep.append(t0)
    return np.array(keep, dtype=np.float32)

def build_cache(pairs_list, x_path, y_path, max_windows: int, tag: str, target_pos_frac: float):
    Xmm = init_memmap_x(x_path, max_windows)
    ymm = init_memmap_y(y_path, max_windows)

    target_pos = int(round(max_windows*target_pos_frac))
    target_neg = max_windows - target_pos

    n=0; pos=0; neg=0
    tstart=time.time()

    for (sub, runid, eeg_path, ecg_path) in tqdm(pairs_list, desc=f"Build {tag} cache"):
        if n>=max_windows: break
        if pos>=target_pos and neg>=target_neg: break

        evf = eeg_path.replace("_eeg.edf","_events.tsv")
        intervals = seizure_intervals(read_events_tsv(evf)) if os.path.exists(evf) else []
        has_sz = len(intervals)>0

        try:
            raw_eeg = preload_guard_read_raw(eeg_path)
            raw_ecg = preload_guard_read_raw(ecg_path)
        except Exception:
            continue

        try:
            raw_eeg_f = maybe_filter(raw_eeg, "eeg")
            raw_ecg_f = maybe_filter(raw_ecg, "ecg")
        except Exception:
            raw_eeg.close(); raw_ecg.close()
            continue

        dur = min(raw_eeg_f.n_times/SFREQ, raw_ecg_f.n_times/SFREQ)
        if dur <= WIN_SEC:
            raw_eeg.close(); raw_ecg.close()
            if raw_eeg_f is not raw_eeg: raw_eeg_f.close()
            if raw_ecg_f is not raw_ecg: raw_ecg_f.close()
            continue

        eeg_picks = pick_eeg_picks(raw_eeg_f)
        ecg_pick = pick_ecg_pick(raw_ecg_f)

        def get_seg(raw, picks, s0):
            if raw.preload:
                return raw._data[picks, s0:s0+WIN_SAMP]
            return raw.get_data(picks=picks, start=s0, stop=s0+WIN_SAMP)

        pos_times = sample_pos_times(intervals, dur)
        bg_times = sample_bg_times(intervals, dur)

        if has_sz:
            if len(pos_times) > POS_CAP_PER_RUN:
                pos_times = np.random.choice(pos_times, size=POS_CAP_PER_RUN, replace=False).astype(np.float32)
            if len(bg_times) > NEG_CAP_PER_RUN_WITH_SEIZURE:
                bg_times = np.random.choice(bg_times, size=NEG_CAP_PER_RUN_WITH_SEIZURE, replace=False).astype(np.float32)
        else:
            if len(bg_times) > NEG_CAP_PER_SEIZURE_FREE_RUN:
                bg_times = np.random.choice(bg_times, size=NEG_CAP_PER_SEIZURE_FREE_RUN, replace=False).astype(np.float32)

        if has_sz and pos < target_pos:
            for t0s in pos_times:
                if n>=max_windows or pos>=target_pos: break
                s0=int(round(float(t0s)*SFREQ))
                eeg_w=get_seg(raw_eeg_f, eeg_picks, s0).astype(np.float32)
                ecg_w=get_seg(raw_ecg_f, [ecg_pick], s0).astype(np.float32)
                if eeg_w.shape[0] < EEG_CH:
                    eeg_w=np.concatenate([eeg_w, np.zeros((EEG_CH-eeg_w.shape[0], WIN_SAMP), dtype=np.float32)], axis=0)
                else:
                    eeg_w=eeg_w[:EEG_CH]
                Xmm[n,:EEG_CH,:]=zscore_clip(eeg_w)
                Xmm[n,EEG_CH:EEG_CH+1,:]=zscore_clip(ecg_w)
                ymm[n]=1
                n+=1; pos+=1

        if neg < target_neg:
            for t0s in bg_times:
                if n>=max_windows or neg>=target_neg: break
                s0=int(round(float(t0s)*SFREQ))
                eeg_w=get_seg(raw_eeg_f, eeg_picks, s0).astype(np.float32)
                ecg_w=get_seg(raw_ecg_f, [ecg_pick], s0).astype(np.float32)
                if eeg_w.shape[0] < EEG_CH:
                    eeg_w=np.concatenate([eeg_w, np.zeros((EEG_CH-eeg_w.shape[0], WIN_SAMP), dtype=np.float32)], axis=0)
                else:
                    eeg_w=eeg_w[:EEG_CH]
                Xmm[n,:EEG_CH,:]=zscore_clip(eeg_w)
                Xmm[n,EEG_CH:EEG_CH+1,:]=zscore_clip(ecg_w)
                ymm[n]=0
                n+=1; neg+=1

        raw_eeg.close(); raw_ecg.close()
        if raw_eeg_f is not raw_eeg: raw_eeg_f.close()
        if raw_ecg_f is not raw_ecg: raw_ecg_f.close()
        gc.collect()

    Xmm.flush(); ymm.flush()
    meta={"windows": int(n), "pos": int(pos), "neg": int(neg), "tag": tag}
    print(f"[{tag}] DONE windows={n} pos={pos} neg={neg} ({(time.time()-tstart)/60:.1f} min)")
    return meta

print("\n=== STEP 1/7: BUILD TRAIN CACHE ===")
train_meta = build_cache(train_pairs, TR_X_MM, TR_Y_MM, TRAIN_MAX_WINDOWS, "train", TARGET_POS_FRAC_TRAIN)

print("\n=== STEP 2/7: BUILD VAL CACHE ===")
val_meta = build_cache(val_pairs, VA_X_MM, VA_Y_MM, VAL_MAX_WINDOWS, "val", TARGET_POS_FRAC_VAL)

N_TR = int(train_meta["windows"])
N_VA = int(val_meta["windows"])

Xtr = np.memmap(TR_X_MM, mode="r", dtype="float32", shape=(TRAIN_MAX_WINDOWS, C_TOTAL, WIN_SAMP))
ytr = np.memmap(TR_Y_MM, mode="r", dtype="uint8",   shape=(TRAIN_MAX_WINDOWS,))
Xva = np.memmap(VA_X_MM, mode="r", dtype="float32", shape=(VAL_MAX_WINDOWS,   C_TOTAL, WIN_SAMP))
yva = np.memmap(VA_Y_MM, mode="r", dtype="uint8",   shape=(VAL_MAX_WINDOWS,))

# =========================
# SEQ DATASET
# =========================
class MemmapSeqDataset(Dataset):
    def __init__(self, X, y, n, seq_len: int, drop_ambiguous: bool):
        self.X = X; self.y = y; self.n = n; self.seq_len = seq_len
        self.starts = np.arange(0, max(0, n - seq_len), dtype=np.int64)

        if drop_ambiguous:
            yy = y[:n].astype(np.float32)
            c = np.cumsum(yy)
            keep=[]
            for s in self.starts:
                e = s + seq_len
                pos_cnt = c[e-1] - (c[s-1] if s > 0 else 0.0)
                frac = float(pos_cnt / seq_len)
                if frac >= SEQ_POS_LAST_PROB or frac <= SEQ_NEG_LAST_PROB:
                    keep.append(s)
            self.starts = np.array(keep, dtype=np.int64)

        if len(self.starts) == 0:
            raise RuntimeError("No valid sequences found. Try DROP_AMBIGUOUS_SEQS=False or relax thresholds.")

    def __len__(self): return len(self.starts)

    def __getitem__(self, idx):
        s = int(self.starts[idx]); e = s + self.seq_len
        x_seq = np.array(self.X[s:e], copy=True)  # (S,C,T)
        y_seq = self.y[s:e].astype(np.float32)
        y_label = 1.0 if float(y_seq.mean()) >= SEQ_POS_LAST_PROB else 0.0
        return torch.from_numpy(x_seq), torch.tensor(y_label, dtype=torch.float32)

def collate_seq(batch):
    xs = torch.stack([b[0] for b in batch], dim=0)  # (B,S,C,T)
    ys = torch.stack([b[1] for b in batch], dim=0)  # (B,)
    return xs, ys

train_ds = MemmapSeqDataset(Xtr, ytr, N_TR, SEQ_LEN, DROP_AMBIGUOUS_SEQS)
val_ds   = MemmapSeqDataset(Xva, yva, N_VA, SEQ_LEN, False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, collate_fn=collate_seq)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, collate_fn=collate_seq)

# =========================
# MODEL
# =========================
class WindowCNN(nn.Module):
    def __init__(self, in_ch: int, base: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, base, 9, padding=4),
            nn.BatchNorm1d(base), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(base, base*2, 7, padding=3),
            nn.BatchNorm1d(base*2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(base*2, base*4, 5, padding=2),
            nn.BatchNorm1d(base*4), nn.ReLU(),
            nn.Conv1d(base*4, base*4, 3, padding=1),
            nn.BatchNorm1d(base*4), nn.ReLU(),
        )
        self.out_dim = base*4
    def forward(self, x):
        return self.net(x).mean(dim=-1)

class CNNBiLSTM(nn.Module):
    def __init__(self, in_ch: int, feat_base: int = 64, lstm_hidden: int = 128, lstm_layers: int = 1):
        super().__init__()
        self.cnn = WindowCNN(in_ch, base=feat_base)
        self.lstm = nn.LSTM(
            input_size=self.cnn.out_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.0 if lstm_layers == 1 else 0.2,
        )
        self.head = nn.Sequential(
            nn.Linear(2*lstm_hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(128, 1)
        )
    def forward(self, x_seq):  # (B,S,C,T)
        B,S,C,T = x_seq.shape
        x_flat = x_seq.view(B*S, C, T)
        f = self.cnn(x_flat).view(B, S, -1)
        out, _ = self.lstm(f)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

model = CNNBiLSTM(in_ch=C_TOTAL, feat_base=64, lstm_hidden=128, lstm_layers=1).to(DEVICE)
print("\n=== STEP 3/7: TRAIN CNN+BiLSTM (sequence training) ===")
print(model)

# pos_weight on sequences
y_train_seq=[]
for i in range(min(5000, len(train_ds))):
    _, yy = train_ds[i]
    y_train_seq.append(float(yy.item()))
y_train_seq = np.array(y_train_seq) if len(y_train_seq) else np.array([0,1])
p_seq = float((y_train_seq == 1).mean())
pos_weight = torch.tensor([max(1.0, (1.0 - p_seq) / max(1e-6, p_seq))], device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_score=-1e9; bad=0
best_Y=None; best_P=None

for ep in range(MAX_EPOCHS):
    model.train()
    tot=0.0
    for xb,yb in tqdm(train_loader, desc=f"Train {ep+1}/{MAX_EPOCHS}", leave=False):
        xb=xb.to(DEVICE, non_blocking=True)
        yb=yb.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        logits=model(xb)
        loss=criterion(logits, yb)
        loss.backward()
        opt.step()
        tot += float(loss.item())
    train_loss = tot/max(1,len(train_loader))

    model.eval()
    vloss_sum=0.0
    P=[]; Y=[]
    with torch.no_grad():
        for xb,yb in val_loader:
            xb=xb.to(DEVICE, non_blocking=True)
            yb=yb.to(DEVICE, non_blocking=True)
            logits=model(xb)
            vloss_sum += float(criterion(logits, yb).item())
            p=torch.sigmoid(logits).detach().cpu().numpy()
            P.append(p); Y.append(yb.detach().cpu().numpy().astype(int))
    val_loss=vloss_sum/max(1,len(val_loader))
    P=np.concatenate(P); Y=np.concatenate(Y)
    val_aupr=float(average_precision_score(Y,P))
    val_auroc=float(roc_auc_score(Y,P))
    score = val_aupr - SCORE_LOSS_WEIGHT*val_loss
    print(f"Epoch {ep+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_AUPR={val_aupr:.4f} val_AUROC={val_auroc:.4f} score={score:.4f}")

    if score > best_score + MIN_DELTA:
        best_score=score; bad=0
        torch.save(model.state_dict(), MODEL_PATH)
        best_Y = Y.copy(); best_P = P.copy()
        print("  saved best:", MODEL_PATH)
    else:
        bad += 1
        if bad >= PATIENCE:
            print("Early stopping")
            break

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# ===== SAVE SEQ METRICS + CURVES (this is what fixes your paper row) =====
val_seq_auroc = float(roc_auc_score(best_Y, best_P))
val_seq_aupr  = float(average_precision_score(best_Y, best_P))
np.savez_compressed(SEQ_METRICS_NPZ, val_seq_auroc=val_seq_auroc, val_seq_aupr=val_seq_aupr)

fpr, tpr, roc_thr = roc_curve(best_Y, best_P)
prec, rec, pr_thr = precision_recall_curve(best_Y, best_P)
np.savez_compressed(SEQ_ROC_NPZ, fpr=fpr.astype(np.float32), tpr=tpr.astype(np.float32), thr=roc_thr.astype(np.float32))
np.savez_compressed(SEQ_PR_NPZ,  precision=prec.astype(np.float32), recall=rec.astype(np.float32), thr=pr_thr.astype(np.float32))

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, linewidth=2, label=f"SEQ AUROC={val_seq_auroc:.3f}")
plt.plot([0,1],[0,1],"--",alpha=0.5)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Sequence-level ROC")
plt.grid(True, alpha=0.3); plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig(SEQ_ROC_PNG, dpi=220); plt.close()

plt.figure(figsize=(6,5))
plt.plot(rec, prec, linewidth=2, label=f"SEQ AUPR={val_seq_aupr:.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Sequence-level Precision-Recall")
plt.grid(True, alpha=0.3); plt.legend(loc="lower left"); plt.tight_layout()
plt.savefig(SEQ_PR_PNG, dpi=220); plt.close()

print("\n=== STEP 4/7: FINAL SEQ-VAL METRICS (saved to OUT_DIR) ===")
print("VAL SEQ AUROC:", val_seq_auroc)
print("VAL SEQ AUPR :", val_seq_aupr)
print("Saved:", SEQ_METRICS_NPZ)
print("Saved:", SEQ_ROC_NPZ)
print("Saved:", SEQ_PR_NPZ)
print("Saved:", SEQ_ROC_PNG)
print("Saved:", SEQ_PR_PNG)

# =========================
# STEP 5/7: CACHE VAL RUN PROBS
# =========================
print("\n=== STEP 5/7: CACHE VAL RUN PROBS (CNN+BiLSTM inference) ===")

@torch.no_grad()
def compute_run_probs(sub: str, runid: str, eeg_path: str, ecg_path: str) -> str:
    out_npz = os.path.join(VAL_RUN_PROB_DIR, f"{sub}_run-{runid}_cnn_bilstm.npz")
    if os.path.exists(out_npz):
        return out_npz

    evf = eeg_path.replace("_eeg.edf","_events.tsv")
    intervals = seizure_intervals(read_events_tsv(evf)) if os.path.exists(evf) else []
    intervals_arr = np.array(intervals, dtype=np.float32).reshape(-1,2)

    try:
        raw_eeg = preload_guard_read_raw(eeg_path)
        raw_ecg = preload_guard_read_raw(ecg_path)
    except Exception:
        return ""

    raw_eeg_f = maybe_filter(raw_eeg, "eeg")
    raw_ecg_f = maybe_filter(raw_ecg, "ecg")

    dur = min(raw_eeg_f.n_times/SFREQ, raw_ecg_f.n_times/SFREQ)
    times = np.arange(0.0, max(0.0, dur-WIN_SEC)+1e-9, PRED_STEP_SEC, dtype=np.float32)
    probs = np.zeros((len(times),), dtype=np.float32)

    eeg_picks = pick_eeg_picks(raw_eeg_f)
    ecg_pick = pick_ecg_pick(raw_ecg_f)

    def get_seg(raw, picks, s0):
        if raw.preload:
            return raw._data[picks, s0:s0+WIN_SAMP]
        return raw.get_data(picks=picks, start=s0, stop=s0+WIN_SAMP)

    feat_buf=[]
    F_dim = model.cnn.out_dim

    for i,t0s in enumerate(times):
        s0=int(round(float(t0s)*SFREQ))
        eeg_w=get_seg(raw_eeg_f, eeg_picks, s0).astype(np.float32)
        ecg_w=get_seg(raw_ecg_f, [ecg_pick], s0).astype(np.float32)

        if eeg_w.shape[0] < EEG_CH:
            eeg_w=np.concatenate([eeg_w, np.zeros((EEG_CH-eeg_w.shape[0], WIN_SAMP), dtype=np.float32)], axis=0)
        else:
            eeg_w=eeg_w[:EEG_CH]

        x = np.concatenate([zscore_clip(eeg_w), zscore_clip(ecg_w)], axis=0)
        xb = torch.from_numpy(x[None, ...]).to(DEVICE)

        f = model.cnn(xb).detach().cpu().numpy().reshape(-1)
        feat_buf.append(f)
        if len(feat_buf) > SEQ_LEN:
            feat_buf.pop(0)

        feats = feat_buf
        if len(feats) < SEQ_LEN:
            pad = [np.zeros((F_dim,), dtype=np.float32) for _ in range(SEQ_LEN - len(feats))]
            feats = pad + feats

        feats_np = np.stack(feats, axis=0).astype(np.float32)
        feats_t = torch.from_numpy(feats_np[None, ...]).to(DEVICE)
        out, _ = model.lstm(feats_t)
        last = out[:, -1, :]
        logit = model.head(last).squeeze(-1)
        probs[i] = float(torch.sigmoid(logit).item())

    np.savez_compressed(out_npz, times=times, probs=probs, duration_sec=np.float32(dur), intervals=intervals_arr)

    raw_eeg.close(); raw_ecg.close()
    if raw_eeg_f is not raw_eeg: raw_eeg_f.close()
    if raw_ecg_f is not raw_ecg: raw_ecg_f.close()
    gc.collect()
    return out_npz

prob_files=[]
for (sub,runid,eeg_path,ecg_path) in tqdm(val_pairs, desc="VAL run probs"):
    fp = compute_run_probs(sub, runid, eeg_path, ecg_path)
    if fp: prob_files.append(fp)
print("VAL prob files:", len(prob_files))

# =========================
# Alarm helpers
# =========================
def alarm_series(probs: np.ndarray, thr: float, window_sec: int, min_pos: int, step_sec: float) -> np.ndarray:
    hits = (probs >= thr).astype(np.int32)
    k = int(round(window_sec/step_sec))
    if k<=0 or len(hits)<k:
        return np.zeros_like(hits, dtype=bool)
    c = np.cumsum(hits, dtype=np.int32)
    win = c.copy()
    win[k:] = c[k:] - c[:-k]
    win[:k-1] = 0
    return win >= min_pos

def apply_refractory(alarm_bool: np.ndarray, step_sec: float, refractory_sec: float) -> np.ndarray:
    if refractory_sec <= 0:
        return alarm_bool
    k = int(round(refractory_sec/step_sec))
    if k<=0:
        return alarm_bool
    out = alarm_bool.copy()
    diff = np.diff(out.astype(np.int32), prepend=0)
    onsets = np.where(diff==1)[0]
    for s in onsets:
        out[s+1:s+k] = False
    return out

def count_episode_fa(alarm_bool: np.ndarray, in_sz: np.ndarray, step_sec: float, episode_gap_sec: float) -> int:
    bg = alarm_bool & (~in_sz)
    if not bg.any(): return 0
    diff = np.diff(bg.astype(np.int32), prepend=0)
    on = np.where(diff==1)[0]
    if len(on)==0: return 0
    gap = int(round(max(0.0, episode_gap_sec)/step_sec))
    eps=1; last=on[0]
    for o in on[1:]:
        if o-last >= gap:
            eps += 1
            last = o
    return eps

# =========================
# STEP 6/7: EVENT CURVES
# =========================
print("\n=== STEP 6/7: BUILD EVENT CURVES ===")

rows_window=[]
rows_episode=[]

for (wsec,minpos) in ALARM_RULES:
    for thr in tqdm(THRESHOLDS, desc=f"rule {minpos}/{wsec}"):
        det=0; total=0
        bg_pos_points=0
        bg_points=0
        fa_eps=0
        hours=0.0

        for fp in prob_files:
            d=np.load(fp)
            times=d["times"].astype(np.float32)
            probs=d["probs"].astype(np.float32)
            dur=float(d["duration_sec"])
            intervals=d["intervals"]
            intervals=intervals.reshape(-1,2) if intervals.size else np.zeros((0,2), dtype=np.float32)
            if len(times)==0:
                continue

            in_sz=np.zeros_like(times, dtype=bool)
            for a,b in intervals:
                in_sz |= (times>=a) & (times<=b)

            alarm_raw = alarm_series(probs, thr, wsec, minpos, step_sec=PRED_STEP_SEC)

            bg = ~in_sz
            bg_pos_points += int((alarm_raw & bg).sum())
            bg_points += int(bg.sum())

            alarm_fa = apply_refractory(alarm_raw, step_sec=PRED_STEP_SEC, refractory_sec=REFRACTORY_SEC) if FA_USE_REFRACTORY else alarm_raw
            fa_eps += count_episode_fa(alarm_fa, in_sz, step_sec=PRED_STEP_SEC, episode_gap_sec=EPISODE_GAP_SEC)

            got=0
            for a,b in intervals:
                a2 = max(0.0, float(a) - PRE_TOL_SEC)
                m=(times>=a2) & (times<=float(b))
                if m.any() and alarm_raw[m].any():
                    got += 1
            det += got
            total += int(len(intervals))
            hours += dur/3600.0

        sens = det/total if total else np.nan
        window_fah = bg_pos_points / max(1e-9, hours)
        baf = bg_pos_points / max(1, bg_points)
        episode_fah = fa_eps / max(1e-9, hours)

        rows_window.append({
            "window_sec": wsec, "min_pos": minpos, "threshold": float(thr),
            "event_sensitivity": float(sens),
            "fa_per_hour": float(window_fah),
            "bg_alarm_fraction": float(baf),
            "detected_events": int(det), "total_events": int(total),
            "pretol_sec": float(PRE_TOL_SEC),
        })
        rows_episode.append({
            "window_sec": wsec, "min_pos": minpos, "threshold": float(thr),
            "event_sensitivity": float(sens),
            "fa_per_hour": float(episode_fah),
            "bg_alarm_fraction": float(baf),
            "detected_events": int(det), "total_events": int(total),
            "pretol_sec": float(PRE_TOL_SEC),
            "refractory_sec": float(REFRACTORY_SEC),
            "episode_gap_sec": float(EPISODE_GAP_SEC),
        })

dfw=pd.DataFrame(rows_window)
dfe=pd.DataFrame(rows_episode)
dfw.to_csv(CSV_WINDOW, index=False)
dfe.to_csv(CSV_EPISODE, index=False)
print("Saved:", CSV_WINDOW)
print("Saved:", CSV_EPISODE)

# =========================
# STEP 7/7: PAPER PLOTS + POINTWISE METRICS AT BEST EPISODE OP POINT
# =========================
print("\n=== STEP 7/7: PAPER PLOTS + POINTWISE METRICS ===")

# Best episode op point under caps
d_caps = dfe[(dfe["fa_per_hour"] < FAH_LIMIT) & (dfe["bg_alarm_fraction"] <= BG_ALARM_FRACTION_CAP)].copy()
best = d_caps.sort_values(["event_sensitivity","fa_per_hour","threshold"], ascending=[False, True, False]).iloc[0]
print("BEST (episode under caps): sens=", float(best.event_sensitivity), "FA/h=", float(best.fa_per_hour), "thr=", float(best.threshold))

# episode envelope plot (nice)
d = dfe[(dfe["bg_alarm_fraction"] <= BG_ALARM_FRACTION_CAP) & (dfe["fa_per_hour"] <= 100.0)].copy().sort_values("fa_per_hour")
x = d["fa_per_hour"].to_numpy(float)
y = d["event_sensitivity"].to_numpy(float)
bins = np.arange(0, 101, 1.0)
xs=[]; ys=[]
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    m = (x >= lo) & (x < hi)
    if not m.any(): continue
    xs.append((lo+hi)/2)
    ys.append(float(np.max(y[m])))
xs=np.array(xs); ys=np.maximum.accumulate(np.array(ys))

plt.figure(figsize=(7,5))
plt.plot(xs, ys, linewidth=2)
plt.scatter([float(best.fa_per_hour)], [float(best.event_sensitivity)], s=60, color="red", label="Best op point", zorder=3)
plt.xlim(0, 100); plt.ylim(0, 1.02)
plt.xlabel("Episode FA/h"); plt.ylabel("Event sensitivity")
plt.title("Sensitivity vs FA/h (Episode), monotone envelope (BAF<=0.30)")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(EP_ENVELOPE_PNG, dpi=220); plt.close()
print("Saved:", EP_ENVELOPE_PNG)

# pointwise ROC/PR + confusion matrix at best threshold
thr_op = float(best.threshold)

def labels_from_intervals(times, intervals):
    y = np.zeros((len(times),), dtype=np.uint8)
    arr = np.array(intervals, dtype=np.float32)
    if arr.size == 0:
        return y
    arr = arr.reshape(-1,2)
    for a,b in arr:
        y |= ((times >= float(a)) & (times <= float(b))).astype(np.uint8)
    return y

P_all=[]; Y_all=[]
for fp in tqdm(prob_files, desc="Collect pointwise arrays"):
    dd = np.load(fp, allow_pickle=True)
    times = dd["times"].astype(np.float32)
    probs = dd["probs"].astype(np.float32)
    intervals = dd["intervals"]
    if len(times) == 0:
        continue
    ylab = labels_from_intervals(times, intervals)
    P_all.append(probs); Y_all.append(ylab)

P_all = np.concatenate(P_all).astype(np.float32)
Y_all = np.concatenate(Y_all).astype(np.uint8)

pos = int(Y_all.sum()); neg = int((Y_all == 0).sum())
print("Pointwise: timepoints=", len(Y_all), "pos=", pos, "neg=", neg, "pos_frac=", pos/max(1,len(Y_all)))

pw_auroc = float(roc_auc_score(Y_all, P_all))
pw_aupr  = float(average_precision_score(Y_all, P_all))
print("Pointwise AUROC:", pw_auroc)
print("Pointwise AUPR :", pw_aupr)

yhat = (P_all >= thr_op).astype(np.uint8)
cm = confusion_matrix(Y_all, yhat, labels=[0,1])
tn, fp_, fn, tp = cm.ravel()

acc  = float(accuracy_score(Y_all, yhat))
bacc = float(balanced_accuracy_score(Y_all, yhat))
prec = float(precision_score(Y_all, yhat, zero_division=0))
rec  = float(recall_score(Y_all, yhat, zero_division=0))
f1   = float(f1_score(Y_all, yhat, zero_division=0))
spec = float(tn / max(1, tn+fp_))

np.savez_compressed(
    PW_METRICS_NPZ,
    thr=np.float32(thr_op),
    cm=cm.astype(np.int64),
    acc=np.float32(acc),
    bacc=np.float32(bacc),
    precision=np.float32(prec),
    recall=np.float32(rec),
    specificity=np.float32(spec),
    f1=np.float32(f1),
    pointwise_auroc=np.float32(pw_auroc),
    pointwise_aupr=np.float32(pw_aupr),
)
print("Saved:", PW_METRICS_NPZ)

fpr, tpr, _ = roc_curve(Y_all, P_all)
pr_prec, pr_rec, _ = precision_recall_curve(Y_all, P_all)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, linewidth=2, label=f"Pointwise AUROC={pw_auroc:.3f}")
plt.plot([0,1],[0,1],"--",alpha=0.5)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Pointwise ROC (1s timepoints)")
plt.grid(True, alpha=0.3); plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig(PW_ROC_PNG, dpi=220); plt.close()
print("Saved:", PW_ROC_PNG)

plt.figure(figsize=(6,5))
plt.plot(pr_rec, pr_prec, linewidth=2, label=f"Pointwise AUPR={pw_aupr:.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Pointwise Precision-Recall (1s timepoints)")
plt.grid(True, alpha=0.3); plt.legend(loc="lower left"); plt.tight_layout()
plt.savefig(PW_PR_PNG, dpi=220); plt.close()
print("Saved:", PW_PR_PNG)

plt.figure(figsize=(7,4))
plt.hist(P_all, bins=200, alpha=0.85)
plt.axvline(thr_op, color="red", linestyle="--", linewidth=2, label=f"thr={thr_op:.4f}")
plt.yscale("log")
plt.xlabel("Predicted probability"); plt.ylabel("Count (log scale)")
plt.title("Predicted probability distribution (all timepoints)")
plt.grid(True, alpha=0.2); plt.legend(); plt.tight_layout()
plt.savefig(PW_HIST_PNG, dpi=220); plt.close()
print("Saved:", PW_HIST_PNG)

print("\nDONE. All outputs in:", OUT_DIR)
print("Key paper files:")
print(" -", CSV_EPISODE)
print(" -", SEQ_METRICS_NPZ, "(SEQ AUROC/AUPR)")
print(" -", SEQ_ROC_PNG, SEQ_PR_PNG, "(SEQ curves)")
print(" -", EP_ENVELOPE_PNG, "(episode envelope / AUSF)")
print(" -", PW_METRICS_NPZ, PW_ROC_PNG, PW_PR_PNG, "(pointwise metrics/curves)")

DEVICE: cuda
Paired EEG+ECG runs: 2804 | train: 2048 | val: 756

=== STEP 1/7: BUILD TRAIN CACHE ===


Build train cache: 100%|██████████| 2048/2048 [1:02:48<00:00,  1.84s/it]


[train] DONE windows=270574 pos=30574 neg=240000 (62.8 min)

=== STEP 2/7: BUILD VAL CACHE ===


Build val cache: 100%|██████████| 756/756 [21:28<00:00,  1.70s/it]


[val] DONE windows=103916 pos=7916 neg=96000 (21.5 min)

=== STEP 3/7: TRAIN CNN+BiLSTM (sequence training) ===
CNNBiLSTM(
  (cnn): WindowCNN(
    (net): Sequential(
      (0): Conv1d(3, 64, kernel_size=(9,), stride=(1,), padding=(4,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
      (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
      (9): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (10): ReLU()
      (11): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(1,))
      (12): BatchNorm1d(256, eps=1e-05, mom

...

VAL prob files: 756

=== STEP 6/7: BUILD EVENT CURVES ===


rule 24/30: 100%|██████████| 56/56 [01:12<00:00,  1.29s/it]


Saved: /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES/event_curve_window_fah.csv
Saved: /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES/event_curve_episode_fah.csv

=== STEP 7/7: PAPER PLOTS + POINTWISE METRICS ===
BEST (episode under caps): sens= 0.8554216867469879 FA/h= 14.749908269079654 thr= 0.37499999999999994
Saved: /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES/episode_envelope_monotone.png


Collect pointwise arrays: 100%|██████████| 756/756 [00:00<00:00, 796.16it/s]


Pointwise: timepoints= 9911387 pos= 9521 neg= 9901866 pos_frac= 0.0009606122735395157
Pointwise AUROC: 0.8226810910514416
Pointwise AUPR : 0.007097089290176814
Saved: /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES/pointwise_metrics_at_best_episode_thr.npz
Saved: /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES/pointwise_ROC.png
Saved: /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES/pointwise_PR.png
Saved: /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES/pointwise_prob_histogram.png

DONE. All outputs in: /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES
Key paper files:
 - /kaggle/working/seizeit2_PROPOSED_cnn_bilstm_win2_eval_pretol30_det_norefr_fa_refr_WITH_SEQ_SAVES/event_curve_episode_fah.csv
 - /kaggle/working/seizeit2_PROPOSED_cnn_bilstm